In [1]:
import _referAsMain
import torch
from dataset import svg_dataset
from paths_cfg import OUR_DATASET_DIRECTORY, TOKENIZER_SAVE_DIRECTORY
from pathlib import Path
import metrics.chunck as chunck
import tokenizer_pfe.tokenizer_project as tokenizerLib

added '/home/holo/dev/PFE_LLM_art_generation' to import paths


In [2]:
tokenizer_path = TOKENIZER_SAVE_DIRECTORY.joinpath("our_tokenizer")
tokenizer = tokenizerLib.Tokenizer.load(tokenizer_path)

loading the tokenizer from: /home/holo/dev/PFE_LLM_art_generation/tokenizer_save/our_tokenizer.json


In [3]:
dataset = svg_dataset.SVGDataset(
    OUR_DATASET_DIRECTORY, context_size=4096,
    tokenizer=tokenizer.encode, decoder=tokenizer.decode)

In [4]:
for i in range(3):
    print(f"Chunk {i}")
    print(dataset.chunks[i].text[:40])
    print()

Chunk 0
<svg xmlns:xlink="http://www.w3.org/1999

Chunk 1
6197" cy="127.5109"/><circle r="0.5" sty

Chunk 2
line y2="196.6829" style="fill:none;" x1



In [5]:
N = 0
for i in range(10):
    print(" "*N, dataset.chunks[i].text, sep="")
    N += len(dataset.chunks[i].text)//2


<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:'Dialog'; font-style:normal; stroke-linejoin:miter; font-size:12px; stroke-dashoffset:0; image-rendering:auto;" width="900" height="900"><defs id="genericDefs"/><g><g style="stroke-linecap:round; fill:white; stroke:white;"><rect x="0" width="900" height="900" y="0" style="stroke:none;"/></g><g style="stroke-linecap:round;"><circle r="0.5" style="fill:none;" cx="276.6808" cy="156.9013"/><circle r="0.5" style="fill:none;" cx="287.7379" cy="89.1578"/><circle r="0.5" style="fill:none;" cx="304.5989" cy="105.7422"/><circle r="0.5" style="fill:none;" cx="302.5389" cy="115.5454"/><circle r="0.5" style="fill:none;" cx="275.0295" cy="96.4022"/>

In [6]:
#import importlib
#import metrics.chunck
#_ = importlib.reload(metrics.chunck)
#import metrics.chunck as chunck

chunk_tokens = dataset.chunks[0].tokens
#chunk_tokens = torch.tensor(list(range(256)))
seq_len = len(chunk_tokens)
vocab_size = tokenizer.get_vocab_size()
print(seq_len, vocab_size)
chunk_logits = torch.full((1, seq_len, vocab_size), -torch.inf)
#chunk_logits = torch.randn(1, seq_len, vocab_size)
for i, tk in enumerate(chunk_tokens):
    chunk_logits[0, i, tk] = 10.0


tokens_sampled = chunck.sampling_logits(chunk_logits, top_k=None, temperature=1.0)
print("Tokens of original :", chunk_tokens[: 200].tolist())
print("Tokens after sampling :", tokens_sampled[: 200])
print(f"nb diffs: {sum([a!=b for a,b in zip(chunk_tokens.tolist(), tokens_sampled)])}")

4096 585
Tokens of original : [519, 487, 518, 258, 491, 465, 492, 460, 20, 496, 16, 328, 365, 514, 3, 487, 258, 491, 465, 492, 460, 20, 496, 16, 382, 386, 513, 3, 269, 258, 273, 503, 27, 18, 28, 490, 441, 425, 28, 490, 581, 425, 28, 572, 441, 425, 28, 406, 497, 28, 406, 428, 583, 28, 406, 582, 27, 384, 28, 573, 441, 425, 28, 406, 503, 27, 18, 28, 484, 497, 28, 406, 577, 275, 28, 440, 565, 502, 28, 406, 511, 27, 18, 28, 440, 579, 515, 570, 505, 440, 566, 502, 28, 406, 576, 517, 28, 440, 580, 27, 350, 539, 28, 406, 578, 27, 17, 28, 574, 441, 425, 268, 485, 258, 385, 17, 3, 486, 258, 385, 17, 504, 528, 547, 258, 560, 276, 72, 266, 72, 269, 258, 427, 428, 444, 28, 484, 498, 28, 406, 498, 456, 556, 260, 258, 17, 3, 485, 258, 385, 17, 3, 486, 258, 385, 17, 3, 261, 258, 17, 3, 269, 258, 427, 275, 575, 72, 266, 72, 269, 258, 427, 428, 444, 456, 338, 327, 258, 17, 15, 22, 3, 269, 258, 273, 275, 268, 331, 258, 313, 23, 15, 322, 374, 3, 332, 258, 344, 23, 15]
Tokens after sampling : [519, 487, 51

In [ ]:
logits_dict = {(0, 0): chunk_logits}
svgs_text, svg_tokens = chunck.assemble_decode(logits_dict, tokenizer)
print(svg_tokens[0][:2000])
print(list(map(ord, svgs_text[0])))
print(list(map(ord, svgs_text[0])).count(65533), len(svgs_text[0]))
print(f"Text decode: {svgs_text[0][:2000]!r}")

[136, 68, 234, 61, 77, 247, 433, 448, 417, 85, 129, 520, 258, 252, 280, 400, 380, 98, 119, 472, 523, 178, 382, 386, 213, 452, 460, 220, 193, 239, 206, 123, 206, 14, 198, 528, 285, 153, 25, 51, 419, 184, 420, 429, 388, 267, 402, 150, 126, 301, 93, 72, 207, 60, 577, 207, 14, 387, 290, 260, 126, 472, 529, 430, 366, 28, 488, 62, 447, 364, 152, 531, 517, 317, 220, 26, 88, 269, 550, 47, 202, 109, 79, 79, 163, 103, 138, 447, 78, 195, 77, 106, 369, 295, 239, 159, 508, 292, 504, 500, 67, 86, 359, 412, 9, 342, 277, 322, 555, 497, 137, 59, 207, 180, 518, 57, 62, 309, 267, 140, 203, 240, 496, 211, 473, 173, 525, 441, 387, 473, 482, 417, 349, 447, 311, 256, 345, 186, 260, 346, 144, 101, 292, 47, 96, 540, 154, 525, 195, 529, 216, 571, 569, 397, 521, 335, 203, 182, 155, 208, 60, 323, 211, 563, 68, 334, 360, 516, 450, 429, 200, 496, 313, 468, 310, 401, 320, 430, 335, 21, 188, 547, 84, 43, 218, 523, 430, 457, 199, 80, 117, 558, 94, 140, 503, 75, 569, 499, 108, 113, 570, 398, 410, 360, 544, 524, 33, 411

In [25]:
logits_dict = {}
for chunk in dataset.chunks[:5]:
    seq_len = len(chunk.tokens)
    if False:
        # pour tokens random
        chunk_logits = torch.randn(1, seq_len, vocab_size)
    else: # pour les tokens du dataset (100% de probas car -inf)
        chunk_logits = torch.full((1, seq_len, vocab_size), -torch.inf)
        for i, tk in enumerate(chunk.tokens):
            chunk_logits[0, i, tk] = 0.0
    idxs = chunk.indexes
    logits_dict[(idxs.svgIndex, idxs.chunckIndex)] = chunk_logits

svgs_text, svg_tokens = chunck.assemble_decode(logits_dict, tokenizer)
for svg_idx, text in svgs_text.items():
    print(svgs_text[0][:200], "\n")

<svg xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-line 

